# Assignment 3



- Import Libraries

In [276]:
# Loading Libraries
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from gensim.models import FastText
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import time

from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import warnings
import spacy
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning) # Ignore unnecessary warnings from BeautifulSoup

nltk.download('stopwords')
nltk.download('wordnet') 

nlp = spacy.load("en_core_web_sm", disable=["parser", "senter"]) # model for proper noun detection

print("--"*50)
print("Libraries loaded successfully.")
print("--"*50)

[nltk_data] Downloading package stopwords to /home/jaee/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jaee/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


----------------------------------------------------------------------------------------------------
Libraries loaded successfully.
----------------------------------------------------------------------------------------------------


# Part A

### Load dataset

In [277]:
# Loading the dataset
data = pd.read_csv("../data/dataset_with_assignments.csv")

print("--"*50)
print("First 5 Rows of the Dataset:")
print("--"*50)
print(data.head(5))

# Dataset Information
print("--"*50)
print("Dataset Information")
print("--"*50)
print(data.info())

# Checking for missing values
print("--"*50)
print("Missing Values in Each Column:")
print("--"*50)
print(data.isnull().sum())


----------------------------------------------------------------------------------------------------
First 5 Rows of the Dataset:
----------------------------------------------------------------------------------------------------
   page_id                                                url  \
0        1                   http://0769sme.org/index-16.html   
1        4  http://aastocks.com/en/cnhk/quote/quick-quote....   
2        6  http://ada.untergrund.net/?p=boardthread&id=18...   
3        7          http://adrienedurand.wikidot.com/blog:127   
4        8  http://afrafrontpagenews.blogspot.com/2012/03/...   

                           domain  tld                  date  word_count  \
0                     0769sme.org  org  2025-12-04T20:51:02Z        2377   
1                    aastocks.com  com  2025-12-04T21:23:33Z        1738   
2              ada.untergrund.net  net  2025-12-04T20:53:54Z        3759   
3       adrienedurand.wikidot.com  com  2025-12-04T21:11:53Z        1328  

### Preprocessing
- Comparing to previous assignment with TF-IDF, we made the regex condition to be stricter, since we can't rely too much on removal by min_df and max_df as we did for TF-IDF and accidently remove semantically meaningfull words. Also, Word2Vec and FastText are more sensitive to noisy vocabularies, so we need better preprocessing for this case.
- Proper noun filtering using spaCy was considered at first, but it takes about 20min, so commented out.

In [278]:
## Use spaCy for proper noun filtering
#def build_proper_noun_set_spacy(text):
#    doc = nlp(text[:100000])  # spaCy has a character limit
#    return {token.text.lower() for token in doc
#            if token.pos_ == "PROPN"
#            and not token.is_upper  # exclude acronyms like API, CPU
#            and len(token.text) > 2}  # exclude short noise like US, UK

In [279]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Remove \n, \r, \t
    text = re.sub(r'[\n\r\t]+', ' ', text)
    
    # Remove URLs and emails
    text = re.sub(r'(?:https?://|ftp://|www\.)\S+|mailto:\S+|tel:\S+', '', text)
    text = re.sub(r'\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b', '', text)

    # Remove HTML tags like <body>, but first remove script/style content
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text = soup.get_text(separator=' ')

    # Build proper noun set
    # proper_nouns = build_proper_noun_set_spacy(text)

    # Split camelCases
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # Remove proper nouns
    # text = ' '.join(w for w in text.split() if w.lower() not in proper_nouns)

    # Remove separators such as _, -, /, |, ...
    text = re.sub(r'[_\-/|]', ' ', text)
    
    # Lowercase
    text = text.lower()
        
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]', ' ', text)

    # Normalize repeated characters like 'yessss', 'noooooo', ...
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip() # remove multiple spaces

    # Tokenize and filter with stop words and lemmatization
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if len(t) >= 3
              and len(t) <= 20 # Remove very long words
              and t not in stop_words] # Keep 3 letter words like api, cpu, ...
    
    return tokens

In [280]:
# Apply preprocessing
sentences = data['full_text'].apply(preprocess).tolist()

# Find total number of tokens
total_tokens = sum(len(s) for s in sentences)

# Find average number of tokens per document
avg_tokens = np.mean([len(s) for s in sentences])

print("--"*50)
print("Preprocessing completed. Sample of cleaned text:")
print("--"*50)
print(f"Total number of tokens in the corpus: {total_tokens}")
print(f"Average number of tokens per document: {avg_tokens}")
print(f"First 5 words in First Document: {sentences[0][:5]}")  # Print first 5 tokens of the first document
print(f"First 5 words in Second Document: {sentences[1][:5]}")  # Print first 5 tokens of the second document

----------------------------------------------------------------------------------------------------
Preprocessing completed. Sample of cleaned text:
----------------------------------------------------------------------------------------------------
Total number of tokens in the corpus: 4544595
Average number of tokens per document: 719.7648083623693
First 5 words in First Document: ['best', 'resume', 'template', 'ready', 'download']
First 5 words in Second Document: ['stock', 'connect', 'quick', 'quote', 'market']


### Hyperparameter Selection

- vector_size=100
  - rule of thumb: (vocab size of each model=20,724)^(1/4) ≈ 12
  - 100 is way above minimum of 12

- window
  - CBOW: window=5 to capture semantical meaning instead of very tight windows like 2 or 3.
  - Skip-gram: window=3 since Skip-gram is more sensitive to noises.

- epochs
  - epochs=20 for CBOW since they are faster.
  - epochs=10 for Skip-gram since they were slower. 10 is still reasonable amount

- min_count=5
  - Vocab: 39,675, avg occurrences per word: 3,613,647/39,675 ≈ 91
  - The final output showed significant noisy words, especially on Skip-gram (Word2Vec Skip-gram for 'product' gives aster, bulkbuy, amplifier, rhodium, gogogate, ...)
  - We will increase min_count

- min_count=10
  - Vocab: 20,724, avg occurrences per word: 3,613,647/20,724 ≈ 174 (almost twice as bettwer)
  - The finaly output showed less noisy words
  - we will stay with min_count=10

In [281]:
param_cbow = dict(
    vector_size=100,   # embedding dimensions (no need to go high for 6,000 docs)
    window=5,          # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=10,      # ignore words with freq < 10 (remove noisy vocabs)
    sg=0,              # 0 = CBOW, 1 = Skip-gram
    workers=4,         # parallel threads
    epochs=20          # training iterations (CBOW is faster, so set it higher)
    )

param_skipgram = dict(
    vector_size=100,       # embedding dimensions (no need to go high for 6,000 docs)
    window=3,              # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=10,          # ignore words with freq < 10 (remove noisy vocabs)
    sg=1,                  # 0 = CBOW, 1 = Skip-gram
    workers=4,             # parallel threads
    epochs=10,             # training iterations (Skip-gram is slower, so set it lower)
)

- Word2Vec Training - CBOW

In [282]:
start_time = time.time()

model_cbow = Word2Vec(sentences=sentences, **param_cbow)

time_cbow = round(time.time() - start_time, 4)
print(f"Elapsed time for Word2Vec CBOW: {time_cbow:.4f} seconds")

Elapsed time for Word2Vec CBOW: 23.5055 seconds


- Word2Vec Training - Skip-gram

In [283]:
start_time = time.time()

model_skipgram = Word2Vec(sentences=sentences, **param_skipgram)

time_skipgram = round(time.time() - start_time, 4)
print(f"Elapsed time for Word2Vec Skip-gram: {time_skipgram:.4f} seconds")

Elapsed time for Word2Vec Skip-gram: 30.8909 seconds


- FastText Training - CBOW

In [284]:
start_time = time.time()

model_fasttext_cbow = FastText(sentences=sentences, 
                               **param_cbow,
                               min_n=3, # min char n-gram size
                               max_n=6, # max char n-gram size
                               )

time_ft_cbow = round(time.time() - start_time, 4)
print(f"Elapsed time for FastText CBOW: {time_ft_cbow:.4f} seconds")

Elapsed time for FastText CBOW: 112.2977 seconds


- FastText Training - Skip-gram

In [285]:
start_time = time.time()

model_fasttext_skipgram = FastText(sentences=sentences, 
                                   **param_skipgram,
                                    min_n=3, # min char n-gram size
                                    max_n=6, # max char n-gram size
                                   )

time_ft_skipgram = round(time.time() - start_time, 4)
print(f"Elapsed time for FastText Skip-gram: {time_ft_skipgram:.4f} seconds")

Elapsed time for FastText Skip-gram: 63.3590 seconds


- Embedding Exploration

In [286]:
# Print vocabulary size and embedding dimensions
training_summary = pd.DataFrame({
    'Model':       ['Word2Vec CBOW', 'Word2Vec Skip-gram', 'FastText CBOW', 'FastText Skip-gram'],
    'Time (s)':    [time_cbow, time_skipgram, time_ft_cbow, time_ft_skipgram],
    'Vocab Size':  [len(m.wv) for m in [model_cbow, model_skipgram, model_fasttext_cbow, model_fasttext_skipgram]],
    'Vector Size': [m.vector_size for m in [model_cbow, model_skipgram, model_fasttext_cbow, model_fasttext_skipgram]],
})

print(training_summary)

                Model  Time (s)  Vocab Size  Vector Size
0       Word2Vec CBOW   23.5055       27410          100
1  Word2Vec Skip-gram   30.8909       27410          100
2       FastText CBOW  112.2977       27410          100
3  FastText Skip-gram   63.3590       27410          100


In [287]:
# Find top=n similar words for given query words in a model
def find_similar_words(model, query_words, topn=10):
    results = []
    for word in query_words:
        if word in model.wv:
            similar = model.wv.most_similar(word, topn=topn)
            for rank, (similar_word, score) in enumerate(similar, 1):
                results.append({
                    'query': word,
                    'rank':  rank,
                    'similar_word': similar_word,
                    'score': round(score, 4)
                })
        else:
            print(f"Warning: '{word}' not in vocabulary")
    return pd.DataFrame(results)

model_order = ['Word2Vec CBOW', 'Word2Vec Skip-gram', 'FastText CBOW', 'FastText Skip-gram']

# Pivot: rows = rank, columns = each model's (similar_word, score)
def compare_query_word(df_all, query_word):
    subset = df_all[df_all['query'] == query_word].copy()
    
    # Build a multi-column df per model
    frames = {}
    for model_name in model_order:
        m = subset[subset['model'] == model_name][['rank', 'similar_word', 'score']]
        m = m.set_index('rank')
        m.columns = [model_name, 'score']
        frames[model_name] = m
    
    combined = pd.concat(frames.values(), axis=1)
    return combined

In [288]:
# Define query words for each categories
categories = {
    'ECOMMERCE':  ['price', 'product', 'shipping'],
    'GOVERNMENT': ['legislation', 'regulation', 'official'],
    'TECHNICAL':  ['software', 'hardware', 'development'],
    'NEWS':       ['report', 'news', 'coverage'],
    'BLOG':       ['comment', 'blog', 'post'],
    'EDUCATIONAL':['student', 'curriculum', 'academic'],
    'OTHER' :     ['contact', 'event', 'forum'] # OTHER category is expected to be noisy since it is any generic document that doesn't fit the above.
}
query_words = [word for words in categories.values() for word in words]

# Collect results from all models
models_dict = {
    'Word2Vec CBOW': model_cbow,
    'Word2Vec Skip-gram': model_skipgram,
    'FastText CBOW': model_fasttext_cbow,
    'FastText Skip-gram': model_fasttext_skipgram
}

all_dfs = []
for model_name, model in models_dict.items():
    df = find_similar_words(model, query_words, topn=10)
    df['model'] = model_name
    all_dfs.append(df)

df_all = pd.concat(all_dfs, ignore_index=True)

for category, words in categories.items():
    print(f"\n{'='*80}")
    print(f"  CATEGORY: {category}")
    print(f"{'='*80}")
    for word in words:
        if word in df_all['query'].values:
            print("-" * 80)
            print(f"  Query: '{word}'")
            print("-" * 80)
            result = compare_query_word(df_all, word)
            print(result.to_string())



  CATEGORY: ECOMMERCE
--------------------------------------------------------------------------------
  Query: 'price'
--------------------------------------------------------------------------------
     Word2Vec CBOW   score Word2Vec Skip-gram   score       FastText CBOW   score  FastText Skip-gram   score
rank                                                                                                         
1            stock  0.5567                rrp  0.7090             ecprice  0.8930              priced  0.7611
2         discount  0.5475           pidilite  0.6821              priced  0.8339             ecprice  0.7327
3              qty  0.5375           autosell  0.6228             prickly  0.5749                 rrp  0.6237
4         quantity  0.5266             resell  0.6222                 qty  0.5674            pidilite  0.6148
5             msrp  0.5076     alphabetically  0.6211             pricing  0.5521  emailbuildandprice  0.6032
6           priced  0.4922  

# Part B

- Document Vector Construction

- Classification

- Evaluation and Comparison

- Prediction